In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/mitacs_runs"

CASES = {
    "A  original instance":            "A",
    "B  routing alternative":          "B",
    "C  persistent + repair time":     "C",
    "D  routing + repair time":        "D",
    "E  persistent, availability only":"E",
}

PPO_MEAN = "PPO (mean over seeds)"
PPO_SD   = "PPO (sd across seeds)"
BASE_ST  = "base-stock (mean)"

rows, missing = [], []

for label, folder in CASES.items():
    p1 = os.path.join(BASE, folder, "results_comparison.csv")
    p2 = os.path.join(BASE, folder, "stress_test_results.csv")
    if not (os.path.exists(p1) and os.path.exists(p2)):
        missing.append(folder)
        continue

    rc = pd.read_csv(p1, index_col=0)
    st = pd.read_csv(p2)

    ppo_J = float(rc.loc["policy_cost_J", PPO_MEAN])
    bs_J  = float(rc.loc["policy_cost_J", BASE_ST])

    by_policy = st.groupby("policy").agg(
        J=("policy_cost_J", "mean"),
        SL=("mean_service_level", "mean"),
        rec=("recovery", "mean"),
        cens=("censored", "mean"),
    )
    seeds = [i for i in by_policy.index if str(i).startswith("PPO seed")]
    sJ = by_policy.loc[seeds, "J"]
    s_bs = by_policy.loc["base-stock", "J"]

    rows.append({
        "case": label,
        "rout_PPO_J":     round(ppo_J),
        "rout_PPO_sd":    round(float(rc.loc["policy_cost_J", PPO_SD])),
        "rout_base_J":    round(bs_J),
        "rout_gap_pct":   round(100 * (ppo_J - bs_J) / bs_J, 1),
        "rout_PPO_SL":    round(float(rc.loc["mean_service_level", PPO_MEAN]), 3),
        "rout_base_SL":   round(float(rc.loc["mean_service_level", BASE_ST]), 3),
        "rout_PPO_exp":   round(float(rc.loc["expired_units", PPO_MEAN]), 1),
        "rout_base_exp":  round(float(rc.loc["expired_units", BASE_ST]), 1),
        "disr_PPO_J":     round(sJ.mean()),
        "disr_PPO_sd":    round(sJ.std(ddof=1)),
        "disr_base_J":    round(s_bs),
        "disr_gap_pct":   round(100 * (sJ.mean() - s_bs) / s_bs, 1),
        "disr_PPO_SL":    round(by_policy.loc[seeds, "SL"].mean(), 3),
        "disr_base_SL":   round(by_policy.loc["base-stock", "SL"], 3),
        "disr_recovery":  round(by_policy.loc[seeds, "rec"].mean(), 2),
        "disr_never_pct": round(100 * by_policy.loc[seeds, "cens"].mean(), 1),
    })

if missing:
    print("MISSING FOLDERS:", missing, "\n")

T = pd.DataFrame(rows).set_index("case")
T.to_csv(os.path.join(BASE, "case_comparison.csv"))

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)

print("ROUTINE CONDITIONS")
print(T[[c for c in T.columns if c.startswith("rout")]].to_string())

print("\nUNDER DISRUPTION")
print(T[[c for c in T.columns if c.startswith("disr")]].to_string())

print("\nA positive gap means PPO is more expensive than base-stock.")

print("\nBETWEEN-SEED STANDARD ERROR (sd / sqrt(5))")
for case, r in T.iterrows():
    print(f"  {case:<34} routine +/- {r['rout_PPO_sd']/5**0.5:>7,.0f}"
          f"   disruption +/- {r['disr_PPO_sd']/5**0.5:>7,.0f}")

print("\nDECOMPOSITION OF THE DISRUPTION GAP")
if all(k in T.index.str[0].tolist() for k in ["A", "E", "C"]):
    gA = T.loc[T.index.str.startswith("A"), "disr_gap_pct"].iloc[0]
    gE = T.loc[T.index.str.startswith("E"), "disr_gap_pct"].iloc[0]
    gC = T.loc[T.index.str.startswith("C"), "disr_gap_pct"].iloc[0]
    print(f"  A -> E   persistent outages alone   {gA:>6.1f}% -> {gE:>6.1f}%   ({gA-gE:+.1f} points)")
    print(f"  E -> C   repair-time information    {gE:>6.1f}% -> {gC:>6.1f}%   ({gE-gC:+.1f} points)")

print("\nwritten to case_comparison.csv")


ROUTINE CONDITIONS
                                  rout_PPO_J  rout_PPO_sd  rout_base_J  rout_gap_pct  rout_PPO_SL  rout_base_SL  rout_PPO_exp  rout_base_exp
case                                                                                                                                        
A  original instance                   21908         5057        21590           1.5        0.815         0.875           5.3           17.6
B  routing alternative                 21563         2900        21590          -0.1        0.812         0.875           7.7           17.6
C  persistent + repair time            24800         2607        26254          -5.5        0.811         0.865           6.5           23.1
D  routing + repair time               26932         2401        26254           2.6        0.812         0.865           9.8           23.1
E  persistent, availability only       27437         5124        26254           4.5        0.807         0.865          12.4          